In [ ]:
# Solutions live in solutions/. Run from the repository root so imports and paths work.
import os, pathlib, sys
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "workshop").is_dir())
os.chdir(_root)
sys.path.insert(0, str(_root))

# Station D: Altair `JupyterChart`

**Good for:** turning any Altair chart into a widget whose selections come
back to Python. **Needs internet:** it loads Vega from a CDN in the browser.

Docs: https://altair-viz.github.io/user_guide/jupyter_chart.html

In [ ]:
import altair as alt
import ipywidgets as widgets
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer(as_frame=True).frame
cancer["diagnosis"] = cancer["target"].map({0: "malignant", 1: "benign"})
cancer[["mean radius", "mean texture", "diagnosis"]].head()

Drag a rectangle on the chart to select:

In [ ]:
brush = alt.selection_interval(name="brush")
chart = (
    alt.Chart(cancer)
    .mark_point()
    .encode(
        x=alt.X("mean radius:Q"),
        y=alt.Y("mean texture:Q"),
        color=alt.condition(brush, "diagnosis:N", alt.value("lightgray")),
    )
    .add_params(brush)
    .properties(width=450, height=320)
)
jchart = alt.JupyterChart(chart)
jchart

The brush value is a dictionary of field ranges:

In [ ]:
jchart.selections.brush.value

### Your task: connect it

Show, in an `Output`, how many selected tumours are malignant and benign,
every time the brush changes. Filter `cancer` with the ranges in
`jchart.selections.brush.value`.

In [ ]:
counts = widgets.Output()


def on_brush(change):
    rows = cancer
    for field, (low, high) in jchart.selections.brush.value.items():
        rows = rows[rows[field].between(low, high)]
    with counts:
        counts.clear_output(wait=True)
        print(f"{len(rows)} selected")
        print(rows["diagnosis"].value_counts().to_string())


jchart.selections.observe(on_brush, ["brush"])
counts